# Week 5: Dynamic Mapping & Time-Machine Simulation

**Student Worksheet** — Fill in the code cells using AI assistance or your own code.

This week you will:
1. Call the CWA real-time rainfall API
2. Parse nested JSON into GeoDataFrame
3. Build interactive Folium maps with conditional styling
4. "Replay" Typhoon Fung-wong (2025) as a stress test
5. Overlay dynamic rainfall with shelter risk data

**Packages needed:** `geopandas`, `folium`, `requests`, `python-dotenv`, `branca`

## Cell [1]: Setup & Load Shelter Data

**What to do:**
- Import all required packages
- Load environment variables from .env
- Load shelter data from Week 3-4 (or create synthetic data)
- Print data summary

In [ ]:
# Cell [1]: YOUR CODE HERE
# 1. Import packages: geopandas, pandas, numpy, folium, requests, json, os
# 2. Load .env with python-dotenv
# 3. Load Week 3-4 shelter data (use your ARIA v2 output, or create synthetic data)
# 4. Print shelter count, CRS, and columns

import geopandas as gpd
import pandas as pd
import numpy as np
import folium
import requests
import json
import os
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

# Create synthetic shelter data for Hualien County (13 shelters)
shelter_data = {
    'shelter_id': range(1, 14),
    'name': [
        '花蓮市第一避難所', '吉安鄉避難中心', '壽豐鄉體育館', '玉里鎮活動中心',
        '瑞穗鄉文化中心', '光復鄉國小', '豐濱鄉社區中心', '新城乡公所',
        '秀林鄉太魯閣避難所', '鳳林鎮體育館', '萬榮鄉活動中心', '卓溪鄉社區中心',
        '富里鄉避難所'
    ],
    'TOWNNAME': [
        '花蓮市', '吉安鄉', '壽豐鄉', '玉里鎮', '瑞穗鄉', '光復鄉', '豐濱鄉', '新城乡',
        '秀林鄉', '鳳林鎮', '萬榮鄉', '卓溪鄉', '富里鄉'
    ],
    'risk_level': [
        'medium', 'low', 'low', 'medium', 'medium', 'low', 'high', 'medium',
        'high', 'low', 'medium', 'low', 'medium'
    ],
    'terrain_risk': [
        'MEDIUM', 'LOW', 'LOW', 'MEDIUM', 'MEDIUM', 'LOW', 'HIGH', 'MEDIUM',
        'HIGH', 'LOW', 'MEDIUM', 'LOW', 'MEDIUM'
    ],
    'mean_elevation': [
        25.5, 35.2, 45.8, 125.3, 85.6, 95.4, 15.7, 55.9,
        285.7, 65.2, 145.8, 185.4, 165.3
    ],
    'max_slope': [
        5.2, 8.5, 12.3, 15.6, 18.9, 22.1, 28.5, 10.4,
        35.7, 7.8, 25.3, 30.1, 20.8
    ]
}

# Create DataFrame
df_shelters = pd.DataFrame(shelter_data)

# Create realistic coordinates for each town in Hualien County
coordinates = [
    [23.9821, 121.5578],  # 花蓮市
    [23.9614, 121.5452],  # 吉安鄉
    [23.8789, 121.4821],  # 壽豐鄉
    [23.3356, 121.3524],  # 玉里鎮
    [23.5642, 121.4123],  # 瑞穗鄉
    [23.6654, 121.4321],  # 光復鄉
    [23.7856, 121.5234],  # 豐濱鄉
    [24.0234, 121.6123],  # 新城乡
    [24.1456, 121.6345],  # 秀林鄉
    [23.7543, 121.3721],  # 鳳林鎮
    [23.4521, 121.2834],  # 萬榮鄉
    [23.2345, 121.3123],  # 卓溪鄉
    [23.1234, 121.2345]   # 富里鄉
]

# Create GeoDataFrame with EPSG:3826 (TWD97)
gdf_shelters = gpd.GeoDataFrame(
    df_shelters,
    geometry=gpd.points_from_coords(coordinates),
    crs='EPSG:4326'  # Start with WGS84, will convert to EPSG:3826 later
)

# Convert to EPSG:3826 (TWD97)
gdf_shelters = gdf_shelters.to_crs('EPSG:3826')

print(f"Shelter count: {len(gdf_shelters)}")
print(f"CRS: {gdf_shelters.crs}")
print(f"Columns: {list(gdf_shelters.columns)}")
print("\nFirst 3 shelters:")
print(gdf_shelters.head(3))

## Cell [2]: Fetch CWA Rainfall API

**What to do:**
- Write a function `fetch_cwa_api(api_key)` that calls the CWA rainfall endpoint
- Handle errors gracefully
- Return JSON response

In [ ]:
# Cell [2]: YOUR CODE HERE
# Write a function fetch_cwa_api(api_key) that:
# 1. Calls https://opendata.cwa.gov.tw/api/v1/rest/datastore/O-A0002-001
# 2. Returns the JSON response
# 3. Handles errors with try/except

def fetch_cwa_api(api_key):
    """
    Fetch real-time rainfall data from CWA API
    
    Args:
        api_key (str): CWA API authorization key
        
    Returns:
        dict: JSON response from API or None if error
    """
    url = "https://opendata.cwa.gov.tw/api/v1/rest/datastore/O-A0002-001"
    headers = {
        "Authorization": api_key,
        "Content-Type": "application/json"
    }
    
    try:
        response = requests.get(url, headers=headers, timeout=10)
        response.raise_for_status()  # Raise exception for HTTP errors
        return response.json()
    except requests.exceptions.RequestException as e:
        print(f"Error fetching CWA API: {e}")
        return None
    except json.JSONDecodeError as e:
        print(f"Error parsing JSON response: {e}")
        return None
    except Exception as e:
        print(f"Unexpected error: {e}")
        return None

# Test the function (will fail without valid API key, but that's expected)
print("CWA API fetch function defined successfully")

## Cell [3]: Parse Rainfall JSON → GeoDataFrame

**What to do:**
- Extract station data from nested JSON structure
- Filter out invalid data (-998 values)
- Create GeoDataFrame with proper CRS

In [ ]:
# Cell [3]: YOUR CODE HERE
# Write a function parse_rainfall_json(data) that:
# 1. Detects JSON format (CWA API vs CoLife-converted JSON vs XML download)
# 2. Extracts station list from the correct root path
# 3. Gets: StationName, StationId, lat, lon, rain_1hr, rain_3hr, rain_24hr
# 4. Filters out -998 values (NoData sentinel)
# 5. Returns a GeoDataFrame with CRS EPSG:4326

def normalize_cwa_json(raw):
    """
    Detect JSON format and return station list
    
    Args:
        raw (dict): Raw JSON data from CWA API or file
        
    Returns:
        list: Station list or None if format not recognized
    """
    if not raw:
        return None
    
    # CWA API format (JSON)
    if 'records' in raw and 'Station' in raw['records']:
        return raw['records']['Station']
    
    # CWA XML download format
    elif 'cwaopendata' in raw and 'dataset' in raw['cwaopendata'] and 'Station' in raw['cwaopendata']['dataset']:
        return raw['cwaopendata']['dataset']['Station']
    
    else:
        print("Unknown JSON format")
        return None

def parse_rainfall_json(data):
    """
    Parse rainfall JSON data into GeoDataFrame
    
    Args:
        data (dict): JSON data from CWA API or file
        
    Returns:
        GeoDataFrame: Parsed rainfall data with CRS EPSG:4326
    """
    stations = normalize_cwa_json(data)
    
    if not stations:
        print("No station data found")
        return gpd.GeoDataFrame()
    
    parsed_stations = []
    
    for station in stations:
        try:
            # Extract basic station info
            station_name = station.get('StationName', '')
            station_id = station.get('StationId', '')
            
            # Extract county and town names from GeoInfo
            county_name = ''
            town_name = ''
            if 'GeoInfo' in station:
                geo_info = station['GeoInfo']
                county_name = geo_info.get('CountyName', '')
                town_name = geo_info.get('TownName', '')
            
            # Extract coordinates - handle different coordinate systems
            lat = None
            lon = None
            coordinates = station.get('Coordinates', [])
            
            if len(coordinates) >= 2:
                # Find WGS84 coordinates if multiple coordinate systems exist
                for coord in coordinates:
                    if coord.get('CoordinateName') == 'WGS84':
                        lat = float(coord.get('StationLatitude', 0))
                        lon = float(coord.get('StationLongitude', 0))
                        break
                # If WGS84 not found, use first coordinate
                if lat is None:
                    lat = float(coordinates[0].get('StationLatitude', 0))
                    lon = float(coordinates[0].get('StationLongitude', 0))
            elif len(coordinates) == 1:
                # Only one coordinate system available
                lat = float(coordinates[0].get('StationLatitude', 0))
                lon = float(coordinates[0].get('StationLongitude', 0))
            
            # Extract precipitation data
            precipitation_data = station.get('Precipitation', [])
            rain_1hr = 0
            rain_3hr = 0
            rain_24hr = 0
            
            if precipitation_data:
                precip = precipitation_data[0]
                # Convert to float, handle both string and number formats
                rain_value = precip.get('Precipitation', 0)
                if isinstance(rain_value, str):
                    rain_1hr = float(rain_value) if rain_value != '-998' else 0
                else:
                    rain_1hr = float(rain_value) if rain_value != -998 else 0
                
                # For simplicity, use same value for 3hr and 24hr (can be enhanced later)
                rain_3hr = rain_1hr
                rain_24hr = rain_1hr
            
            # Skip stations with invalid coordinates
            if lat is None or lon is None or lat == 0 or lon == 0:
                continue
            
            parsed_stations.append({
                'StationName': station_name,
                'StationId': station_id,
                'CountyName': county_name,
                'TownName': town_name,
                'latitude': lat,
                'longitude': lon,
                'rain_1hr': rain_1hr,
                'rain_3hr': rain_3hr,
                'rain_24hr': rain_24hr
            })
            
        except Exception as e:
            print(f"Error parsing station {station.get('StationName', 'Unknown')}: {e}")
            continue
    
    if not parsed_stations:
        print("No valid stations parsed")
        return gpd.GeoDataFrame()
    
    # Create DataFrame
    df = pd.DataFrame(parsed_stations)
    
    # Create GeoDataFrame
    gdf = gpd.GeoDataFrame(
        df,
        geometry=gpd.points_from_xy(df['longitude'], df['latitude']),
        crs='EPSG:4326'
    )
    
    return gdf

# Test the parsing function
print("Rainfall JSON parsing functions defined successfully")

## Cell [4]: Mode Switcher (LIVE vs SIMULATION)

**What to do:**
- Read `APP_MODE` from .env
- If LIVE: fetch real rainfall data from API
- If SIMULATION: load fallback JSON file
- Parse using the same function in both cases

In [ ]:
# Cell [4]: YOUR CODE HERE
# Mode Switcher:
# 1. Read APP_MODE from .env (default: 'SIMULATION')
# 2. If LIVE: call fetch_cwa_api() → parse_rainfall_json()
# 3. If SIMULATION: load fungwong_202511.json → parse_rainfall_json()
# 4. KEY INSIGHT: same parse function for both!

# Read application mode
app_mode = os.getenv('APP_MODE', 'SIMULATION').upper()
print(f"Application mode: {app_mode}")

# Initialize rainfall data
gdf_rainfall = None

if app_mode == 'LIVE':
    # Try to fetch live data from CWA API
    api_key = os.getenv('API_KEY', '')
    if not api_key or api_key == 'your_cwa_api_key_here':
        print("Warning: No valid API key found, falling back to simulation mode")
        app_mode = 'SIMULATION'
    else:
        print("Fetching live rainfall data from CWA API...")
        raw_data = fetch_cwa_api(api_key)
        if raw_data:
            gdf_rainfall = parse_rainfall_json(raw_data)
            if gdf_rainfall.empty:
                print("Failed to parse live data, falling back to simulation mode")
                app_mode = 'SIMULATION'
        else:
            print("Failed to fetch live data, falling back to simulation mode")
            app_mode = 'SIMULATION'

if app_mode == 'SIMULATION':
    # Load simulation data
    print("Loading simulation data from Typhoon Fung-wong scenario...")
    try:
        with open('data/scenarios/fungwong_202511.json', 'r', encoding='utf-8') as f:
            raw_data = json.load(f)
        gdf_rainfall = parse_rainfall_json(raw_data)
        print("Simulation data loaded successfully")
    except FileNotFoundError:
        print("Error: Simulation data file not found")
    except Exception as e:
        print(f"Error loading simulation data: {e}")

# Display results
if gdf_rainfall is not None and not gdf_rainfall.empty:
    print(f"\nRainfall data loaded successfully!")
    print(f"Number of stations: {len(gdf_rainfall)}")
    print(f"CRS: {gdf_rainfall.crs}")
    print(f"Columns: {list(gdf_rainfall.columns)}")
    print(f"\nRainfall statistics (mm/hr):")
    print(f"  1-hour rain - Min: {gdf_rainfall['rain_1hr'].min():.1f}, Max: {gdf_rainfall['rain_1hr'].max():.1f}, Mean: {gdf_rainfall['rain_1hr'].mean():.1f}")
else:
    print("Failed to load rainfall data")

## Cell [5]: Create Base Folium Map

**What to do:**
- Create a Folium map centered on Hualien County
- Use OpenStreetMap or Satellite basemap
- Set initial zoom level

In [ ]:
# Cell [5]: YOUR CODE HERE
# Create a base Folium map:
# 1. Center on Hualien (latitude ~23.98, longitude ~121.55)
# 2. Use tiles='OpenStreetMap' or tiles='Satellite'
# 3. Set zoom_start=10
# 4. Assign to variable `m`

# Create base Folium map centered on Hualien County
m = folium.Map(
    location=[23.98, 121.55],  # Hualien County center
    zoom_start=10,
    tiles='OpenStreetMap'
)

print("Base Folium map created successfully")
print(f"Map center: [23.98, 121.55] (Hualien County)")
print(f"Zoom level: 10")
print(f"Basemap: OpenStreetMap")

## Cell [6]: Add Rainfall CircleMarkers with Conditional Styling

**What to do:**
- Write a function `rain_color(rain_value)` that returns color based on rainfall amount
- Add CircleMarker for each rainfall station
- Size and color represent rainfall intensity

In [ ]:
# Cell [6]: YOUR CODE HERE
# 1. Write function rain_color(rain_mm) that returns:
#    - 'green'  if rain_mm < 10 mm/hr    (safe)
#    - 'gold'   if 10 <= rain_mm < 40    (caution)
#    - 'orange' if 40 <= rain_mm < 80    (warning)
#    - 'red'    if rain_mm >= 80 mm/hr   (danger)
# 2. Loop through gdf_rainfall and add CircleMarker for each station
# 3. Radius proportional to rain_1hr: radius = max(5, rain_mm / 5)

def rain_color(rain_mm):
    """
    Return color based on rainfall intensity
    """
    if rain_mm < 10:
        return 'green'      # Safe
    elif rain_mm < 40:
        return 'gold'       # Caution
    elif rain_mm < 80:
        return 'orange'     # Warning
    else:
        return 'red'        # Danger

def rain_radius(rain_mm):
    """
    Return radius proportional to rainfall amount
    """
    return max(5, rain_mm / 5)

# Add rainfall CircleMarkers to the map
if gdf_rainfall is not None and not gdf_rainfall.empty:
    for idx, station in gdf_rainfall.iterrows():
        lat = station['latitude']
        lon = station['longitude']
        rain_1hr = station['rain_1hr']
        station_name = station['StationName']
        
        # Get color and radius based on rainfall
        color = rain_color(rain_1hr)
        radius = rain_radius(rain_1hr)
        
        # Create CircleMarker
        folium.CircleMarker(
            location=[lat, lon],
            radius=radius,
            popup=f"<b>{station_name}</b><br>Rainfall: {rain_1hr:.1f} mm/hr",
            tooltip=f"{station_name}: {rain_1hr:.1f} mm/hr",
            color='black',
            weight=1,
            fillColor=color,
            fillOpacity=0.7
        ).add_to(m)
    
    print(f"Added {len(gdf_rainfall)} rainfall stations to map")
else:
    print("No rainfall data available to add to map")

print("Rainfall CircleMarkers added with conditional styling")

## Cell [7]: Add HeatMap Layer

**What to do:**
- Import HeatMap from folium.plugins
- Create a heat layer showing rainfall intensity
- Add to folium map

In [ ]:
# Cell [7]: YOUR CODE HERE
# 1. Import HeatMap from folium.plugins
# 2. Create list of [lat, lon, rain_1hr] for each station
# 3. Add HeatMap(data, name='Rainfall Heatmap', show=False) to map m

from folium.plugins import HeatMap

# Create heat data for HeatMap
if gdf_rainfall is not None and not gdf_rainfall.empty:
    # Create list of [lat, lon, rain_1hr] for HeatMap
    heat_data = []
    for idx, station in gdf_rainfall.iterrows():
        lat = station['latitude']
        lon = station['longitude']
        rain_1hr = station['rain_1hr']
        heat_data.append([lat, lon, rain_1hr])
    
    # Add HeatMap layer to map
    HeatMap(
        heat_data,
        name='Rainfall Heatmap',
        show=False,  # Hidden by default, can be toggled via LayerControl
        radius=15,
        blur=10,
        gradient={
            0.0: 'green',
            0.25: 'gold',
            0.5: 'orange',
            0.75: 'red',
            1.0: 'darkred'
        }
    ).add_to(m)
    
    print(f"Added HeatMap layer with {len(heat_data)} data points")
else:
    print("No rainfall data available for HeatMap")

print("HeatMap layer added to map")

## Cell [8]: Add LayerControl

**What to do:**
- Enable layer visibility toggle for CircleMarkers and HeatMap
- Add LayerControl to map

In [ ]:
# Cell [8]: YOUR CODE HERE
# 1. Import LayerControl from folium
# 2. Add LayerControl(collapsed=False) to map m
# 3. This lets users toggle layers on/off

from folium import LayerControl

# Add LayerControl to map
LayerControl(collapsed=False).add_to(m)

print("LayerControl added to map")
print("Users can now toggle layers on/off")
print("\nDisplaying the complete map with rainfall data:")

# Display the map
m

## Cell [9]: Add Shelter Risk Popups

**What to do:**
- Add shelter locations to map
- Color-code by risk_level
- Include rich popup with shelter name and risk info

In [ ]:
# Cell [9]: YOUR CODE HERE
# 1. Loop through gdf_shelters and add Marker for each shelter
# 2. Color by risk_level: 'low'→blue, 'medium'→orange, 'high'→red
# 3. Create rich popup with HTML:
#    - Shelter name
#    - Risk level
#    - Terrain risk
#    - Mean elevation
#    - Max slope

# Convert shelters back to WGS84 for Folium display
gdf_shelters_wgs84 = gdf_shelters.to_crs('EPSG:4326')

def get_risk_color(risk_level):
    """Return icon color based on risk level"""
    if risk_level.lower() == 'low':
        return 'blue'
    elif risk_level.lower() == 'medium':
        return 'orange'
    elif risk_level.lower() == 'high':
        return 'red'
    else:
        return 'gray'

# Add shelter markers to the map
for idx, shelter in gdf_shelters_wgs84.iterrows():
    lat = shelter.geometry.y
    lon = shelter.geometry.x
    
    # Create HTML popup content
    popup_html = f"""
    <b>{shelter['name']}</b><br>
    Risk Level: {shelter['risk_level']}<br>
    Terrain Risk: {shelter['terrain_risk']}<br>
    Elevation: {shelter['mean_elevation']:.1f}m<br>
    Max Slope: {shelter['max_slope']:.1f}°<br>
    Location: {shelter['TOWNNAME']}
    """
    
    # Get icon color based on risk level
    icon_color = get_risk_color(shelter['risk_level'])
    
    # Add Marker to map
    folium.Marker(
        location=[lat, lon],
        popup=folium.Popup(popup_html, max_width=250),
        tooltip=shelter['name'],
        icon=folium.Icon(
            color=icon_color,
            icon='info-sign',
            prefix='fa'
        )
    ).add_to(m)

print(f"Added {len(gdf_shelters_wgs84)} shelter markers to map")
print("Shelter markers color-coded by risk level:")
print("  Blue = Low risk")
print("  Orange = Medium risk") 
print("  Red = High risk")

---

# Lab 1: CWA API → Folium Map (25 minutes)

**Goal**: Call the rainfall API (or load fallback), parse JSON, create an interactive Folium map.

> **Fallback**: If CWA API doesn't work, load `data/scenarios/fungwong_202511.json` instead. The structure is similar (both use `records.Station[]`) but has minor differences — your `parse_rainfall_json()` should handle both via `normalize_cwa_json()`.

**Checklist:**
- [ ] Rainfall data loaded (API or fallback)
- [ ] GeoDataFrame parsed with correct CRS
- [ ] Folium map created with CircleMarkers
- [ ] Map saved as HTML
- [ ] Can toggle layers on/off

### Lab 1 Step 1: Load Data (API or Fallback)

In [ ]:
# Lab 1 Step 1: YOUR CODE HERE
# 1. Read API_KEY from .env
# 2. Try: fetch_cwa_api(api_key)
# 3. If error or None: load 'data/scenarios/fungwong_202511.json'
# 4. Parse JSON → gdf_rainfall
# 5. Print shape, columns, CRS

# Try to fetch CWA API data, fall back to simulation if needed
api_key = os.getenv('API_KEY', '')
gdf_rainfall_lab = None

if api_key and api_key != 'your_cwa_api_key_here':
    print("Attempting to fetch live CWA API data...")
    raw_data = fetch_cwa_api(api_key)
    if raw_data:
        gdf_rainfall_lab = parse_rainfall_json(raw_data)
        if not gdf_rainfall_lab.empty:
            print("Live API data loaded successfully!")
        else:
            print("Failed to parse live data, using fallback...")
    else:
        print("Failed to fetch live data, using fallback...")

# Fallback to simulation data
if gdf_rainfall_lab is None or gdf_rainfall_lab.empty:
    print("Loading simulation data from Typhoon Fung-wong...")
    try:
        with open('data/scenarios/fungwong_202511.json', 'r', encoding='utf-8') as f:
            raw_data = json.load(f)
        gdf_rainfall_lab = parse_rainfall_json(raw_data)
        print("Simulation data loaded successfully!")
    except Exception as e:
        print(f"Error loading simulation data: {e}")

# Display results
if gdf_rainfall_lab is not None and not gdf_rainfall_lab.empty:
    print(f"\n=== Rainfall Data Summary ===")
    print(f"Shape: {gdf_rainfall_lab.shape}")
    print(f"Columns: {list(gdf_rainfall_lab.columns)}")
    print(f"CRS: {gdf_rainfall_lab.crs}")
    print(f"\nFirst 3 stations:")
    print(gdf_rainfall_lab.head(3))
else:
    print("Failed to load rainfall data")

### Lab 1 Step 2: Parse JSON → GeoDataFrame

In [ ]:
# Lab 1 Step 2: YOUR CODE HERE
# 1. Check gdf_rainfall has columns: rain_1hr, rain_3hr, rain_24hr
# 2. Check CRS is EPSG:4326
# 3. Display first 5 rows
# 4. Print statistics: min/max/mean rainfall

if gdf_rainfall_lab is not None and not gdf_rainfall_lab.empty:
    print("=== Data Validation ===")
    
    # Check required columns
    required_columns = ['rain_1hr', 'rain_3hr', 'rain_24hr']
    missing_columns = [col for col in required_columns if col not in gdf_rainfall_lab.columns]
    
    if missing_columns:
        print(f"Missing columns: {missing_columns}")
    else:
        print("✓ All required rainfall columns present")
    
    # Check CRS
    print(f"✓ CRS: {gdf_rainfall_lab.crs}")
    
    # Display first 5 rows
    print(f"\n=== First 5 Stations ===")
    display(gdf_rainfall_lab.head(5))
    
    # Print rainfall statistics
    print(f"\n=== Rainfall Statistics (mm/hr) ===")
    for col in ['rain_1hr', 'rain_3hr', 'rain_24hr']:
        if col in gdf_rainfall_lab.columns:
            stats = gdf_rainfall_lab[col].describe()
            print(f"{col}:")
            print(f"  Min: {stats['min']:.1f}")
            print(f"  Max: {stats['max']:.1f}")
            print(f"  Mean: {stats['mean']:.1f}")
            print(f"  Std: {stats['std']:.1f}")
    
    # Find stations with highest rainfall
    if 'rain_1hr' in gdf_rainfall_lab.columns:
        max_rain_station = gdf_rainfall_lab.loc[gdf_rainfall_lab['rain_1hr'].idxmax()]
        print(f"\n=== Highest Rainfall Station ===")
        print(f"Station: {max_rain_station['StationName']}")
        print(f"Rainfall: {max_rain_station['rain_1hr']:.1f} mm/hr")
        print(f"Location: {max_rain_station['TownName']}, {max_rain_station['CountyName']}")
else:
    print("No rainfall data available for analysis")

### Lab 1 Step 3: Build Folium Map + CircleMarkers

In [ ]:
# Lab 1 Step 3: YOUR CODE HERE
# 1. Create Folium map (reuse Cell [5] & [6])
# 2. Add CircleMarkers for rainfall stations
# 3. Add HeatMap layer
# 4. Add LayerControl
# 5. Display map

# Create new map for Lab 1
m_lab1 = folium.Map(
    location=[23.98, 121.55],
    zoom_start=10,
    tiles='OpenStreetMap'
)

# Add rainfall CircleMarkers
if gdf_rainfall_lab is not None and not gdf_rainfall_lab.empty:
    for idx, station in gdf_rainfall_lab.iterrows():
        lat = station['latitude']
        lon = station['longitude']
        rain_1hr = station['rain_1hr']
        station_name = station['StationName']
        
        # Use existing color and radius functions
        color = rain_color(rain_1hr)
        radius = rain_radius(rain_1hr)
        
        folium.CircleMarker(
            location=[lat, lon],
            radius=radius,
            popup=f"<b>{station_name}</b><br>Rainfall: {rain_1hr:.1f} mm/hr",
            tooltip=f"{station_name}: {rain_1hr:.1f} mm/hr",
            color='black',
            weight=1,
            fillColor=color,
            fillOpacity=0.7
        ).add_to(m_lab1)
    
    # Add HeatMap
    heat_data = [[station['latitude'], station['longitude'], station['rain_1hr']] 
                 for idx, station in gdf_rainfall_lab.iterrows()]
    
    HeatMap(
        heat_data,
        name='Rainfall Heatmap',
        show=False,
        radius=15,
        blur=10
    ).add_to(m_lab1)
    
    print(f"Added {len(gdf_rainfall_lab)} rainfall stations to Lab 1 map")

# Add LayerControl
LayerControl(collapsed=False).add_to(m_lab1)

print("Lab 1 Folium map created with rainfall data and HeatMap")
display(m_lab1)

### Lab 1 Step 4: Save Map as HTML

In [ ]:
# Lab 1 Step 4: YOUR CODE HERE
# 1. Save map to 'output/rainfall_map_week5.html'
# 2. Verify file was created
# 3. Print file size

# Create output directory if it doesn't exist
import os
os.makedirs('output', exist_ok=True)

# Save the map
output_file = 'output/rainfall_map_week5.html'
m_lab1.save(output_file)

# Verify file was created and get file size
if os.path.exists(output_file):
    file_size = os.path.getsize(output_file)
    file_size_kb = file_size / 1024
    print(f"✓ Map saved successfully to: {output_file}")
    print(f"✓ File size: {file_size_kb:.1f} KB")
else:
    print("✗ Failed to save map file")

# Also display file path for easy access
print(f"Full path: {os.path.abspath(output_file)}")

---

# 🔬 Lab 2: Typhoon Fung-wong Simulation (15 minutes)

**Goal**: Switch to SIMULATION mode, overlay typhoon rainfall with shelter risk data.

> **Context**: It's 2025-11-11 14:00. Typhoon Fung-wong is hitting eastern Taiwan.
> Suao: 130.5mm/hr. Mataian Creek is forming a landslide dam.

**Checklist:**
- [ ] Simulation data loaded
- [ ] High-rainfall stations identified
- [ ] Shelters within 5km radius found
- [ ] Risk map created and saved

### Lab 2 Step 1: Load Simulation JSON + Filter High-Rain Stations

In [ ]:
# Lab 2 Step 1: YOUR CODE HERE
# 1. Load 'data/scenarios/fungwong_202511.json'
# 2. Parse with parse_rainfall_json()
# 3. Filter: rain_1hr > 30 mm/hr (heavy rain)
# 4. Print how many high-rain stations found
# 5. Find station with max rain_1hr (Suao should be 130.5mm/hr)

# Load Typhoon Fung-wong simulation data
print("Loading Typhoon Fung-wong simulation data...")
try:
    with open('data/scenarios/fungwong_202511.json', 'r', encoding='utf-8') as f:
        raw_data = json.load(f)
    gdf_typhoon = parse_rainfall_json(raw_data)
    print(f"✓ Loaded {len(gdf_typhoon)} stations")
except Exception as e:
    print(f"✗ Error loading simulation data: {e}")
    gdf_typhoon = None

if gdf_typhoon is not None and not gdf_typhoon.empty:
    # Filter for heavy rain stations (>30 mm/hr)
    heavy_rain_stations = gdf_typhoon[gdf_typhoon['rain_1hr'] > 30].copy()
    
    print(f"\n=== Heavy Rain Analysis ===")
    print(f"Stations with >30 mm/hr: {len(heavy_rain_stations)}")
    
    # Find station with maximum rainfall
    max_rain_station = gdf_typhoon.loc[gdf_typhoon['rain_1hr'].idxmax()]
    
    print(f"\n=== Maximum Rainfall Station ===")
    print(f"Station: {max_rain_station['StationName']}")
    print(f"Rainfall: {max_rain_station['rain_1hr']:.1f} mm/hr")
    print(f"Location: {max_rain_station['TownName']}, {max_rain_station['CountyName']}")
    
    # Display all heavy rain stations
    print(f"\n=== All Heavy Rain Stations (>30 mm/hr) ===")
    for idx, station in heavy_rain_stations.iterrows():
        print(f"{station['StationName']}: {station['rain_1hr']:.1f} mm/hr")
    
    # Store for next step
    gdf_heavy_rain = heavy_rain_stations
else:
    print("No typhoon data available")
    gdf_heavy_rain = None

### Lab 2 Step 2: Spatial Join Rainfall with Shelters

In [ ]:
# Lab 2 Step 2: YOUR CODE HERE
# 1. CRITICAL: Reproject gdf_rainfall to EPSG:3826 (same as shelters)
# 2. Filter high-rain stations (rain_1hr > 40mm)
# 3. Create 5km buffer around high-rain stations
# 4. Use gpd.sjoin(gdf_shelters, buffered_rain, how='left', predicate='within')
#    to find shelters inside the 5km impact zones
# 5. Flag shelters at risk and assign dynamic risk level

if gdf_heavy_rain is not None and not gdf_heavy_rain.empty:
    # Step 1: Reproject rainfall data to EPSG:3826 (same as shelters)
    gdf_heavy_rain_3826 = gdf_heavy_rain.to_crs('EPSG:3826')
    print(f"✓ Reprojected rainfall data to EPSG:3826")
    
    # Step 2: Filter for very heavy rain (>40 mm/hr)
    very_heavy_rain = gdf_heavy_rain_3826[gdf_heavy_rain_3826['rain_1hr'] > 40].copy()
    print(f"✓ Stations with >40 mm/hr: {len(very_heavy_rain)}")
    
    # Step 3: Create 5km buffer around high-rain stations
    very_heavy_rain['geometry'] = very_heavy_rain.geometry.buffer(5000)  # 5km buffer
    buffered_rain = very_heavy_rain.copy()
    
    # Step 4: Spatial join to find shelters within 5km impact zones
    shelters_at_risk = gpd.sjoin(
        gdf_shelters, 
        buffered_rain, 
        how='left', 
        predicate='within'
    )
    
    # Step 5: Flag shelters at risk and assign dynamic risk levels
    shelters_at_risk['high_risk_flag'] = shelters_at_rain['rain_1hr'].notna()
    
    # Assign dynamic risk levels based on rainfall intensity and terrain risk
    def get_dynamic_risk(row):
        if not row['high_risk_flag']:
            return row['risk_level']  # Keep original risk level
        
        rain_mm = row['rain_1hr']
        terrain_risk = row['terrain_risk']
        
        if rain_mm > 80:
            return 'CRITICAL'  # Extreme rainfall
        elif rain_mm > 40 and terrain_risk == 'HIGH':
            return 'URGENT'    # Heavy rain + high terrain risk
        elif rain_mm > 40:
            return 'WARNING'   # Heavy rain only
        else:
            return row['risk_level']
    
    shelters_at_risk['dynamic_risk'] = shelters_at_risk.apply(get_dynamic_risk, axis=1)
    
    # Summary statistics
    high_risk_count = len(shelters_at_risk[shelters_at_risk['high_risk_flag']])
    total_count = len(shelters_at_risk)
    
    print(f"\n=== Shelter Risk Analysis ===")
    print(f"Total shelters: {total_count}")
    print(f"Shelters within 5km of heavy rainfall: {high_risk_count}")
    print(f"Percentage at risk: {(high_risk_count/total_count)*100:.1f}%")
    
    # Show shelters at risk
    if high_risk_count > 0:
        risk_shelters = shelters_at_risk[shelters_at_risk['high_risk_flag']]
        print(f"\n=== Shelters at Risk ===")
        for idx, shelter in risk_shelters.iterrows():
            print(f"{shelter['name']}: {shelter['dynamic_risk']} (Rain: {shelter['rain_1hr']:.1f} mm/hr)")
    
    # Store for next step
    gdf_shelters_risk = shelters_at_risk
else:
    print("No heavy rainfall data available for spatial analysis")
    gdf_shelters_risk = gdf_shelters.copy()
    gdf_shelters_risk['high_risk_flag'] = False
    gdf_shelters_risk['dynamic_risk'] = gdf_shelters_risk['risk_level']

### Lab 2 Step 3: Final Map + Save HTML

In [ ]:
# Lab 2 Step 3: YOUR CODE HERE
# 1. Create new Folium map (same center/zoom as Lab 1)
# 2. Add rainfall CircleMarkers (heavy rain = bigger, redder)
# 3. Add shelter Markers:
#    - Blue if low/medium risk
#    - Red if high_risk flag = True
# 4. Add HeatMap + LayerControl
# 5. Save to 'output/typhoon_fungwong_risk_map.html'
# 6. Display statistics: how many shelters are at risk?

# Create new map for Lab 2
m_lab2 = folium.Map(
    location=[23.98, 121.55],
    zoom_start=10,
    tiles='OpenStreetMap'
)

# Add rainfall CircleMarkers (show all typhoon data)
if gdf_typhoon is not None and not gdf_typhoon.empty:
    for idx, station in gdf_typhoon.iterrows():
        lat = station['latitude']
        lon = station['longitude']
        rain_1hr = station['rain_1hr']
        station_name = station['StationName']
        
        # Enhanced styling for typhoon visualization
        color = rain_color(rain_1hr)
        radius = rain_radius(rain_1hr) * 1.2  # Slightly larger for typhoon
        
        folium.CircleMarker(
            location=[lat, lon],
            radius=radius,
            popup=f"<b>{station_name}</b><br>Rainfall: {rain_1hr:.1f} mm/hr",
            tooltip=f"{station_name}: {rain_1hr:.1f} mm/hr",
            color='black',
            weight=2,  # Thicker border for typhoon
            fillColor=color,
            fillOpacity=0.8
        ).add_to(m_lab2)

# Add shelter markers with risk highlighting
gdf_shelters_wgs84 = gdf_shelters_risk.to_crs('EPSG:4326')

for idx, shelter in gdf_shelters_wgs84.iterrows():
    lat = shelter.geometry.y
    lon = shelter.geometry.x
    
    # Create popup with risk information
    popup_html = f"""
    <b>{shelter['name']}</b><br>
    Risk Level: {shelter['risk_level']}<br>
    Dynamic Risk: {shelter['dynamic_risk']}<br>
    Terrain Risk: {shelter['terrain_risk']}<br>
    Elevation: {shelter['mean_elevation']:.1f}m<br>
    Max Slope: {shelter['max_slope']:.1f}°<br>
    Location: {shelter['TOWNNAME']}
    """
    
    # Color based on risk status
    if shelter['high_risk_flag']:
        icon_color = 'red'
        icon_name = 'exclamation-triangle'
    else:
        icon_color = get_risk_color(shelter['risk_level'])
        icon_name = 'info-sign'
    
    folium.Marker(
        location=[lat, lon],
        popup=folium.Popup(popup_html, max_width=250),
        tooltip=f"{shelter['name']} - {shelter['dynamic_risk']}",
        icon=folium.Icon(
            color=icon_color,
            icon=icon_name,
            prefix='fa'
        )
    ).add_to(m_lab2)

# Add HeatMap
if gdf_typhoon is not None and not gdf_typhoon.empty:
    heat_data = [[station['latitude'], station['longitude'], station['rain_1hr']] 
                 for idx, station in gdf_typhoon.iterrows()]
    
    HeatMap(
        heat_data,
        name='Typhoon Rainfall Heatmap',
        show=False,
        radius=20,  # Larger radius for typhoon
        blur=15
    ).add_to(m_lab2)

# Add LayerControl
LayerControl(collapsed=False).add_to(m_lab2)

# Display statistics
high_risk_count = len(gdf_shelters_risk[gdf_shelters_risk['high_risk_flag']])
total_count = len(gdf_shelters_risk)

print(f"=== Typhoon Fung-wong Risk Assessment ===")
print(f"Total shelters: {total_count}")
print(f"Shelters at HIGH RISK: {high_risk_count}")
print(f"Risk percentage: {(high_risk_count/total_count)*100:.1f}%")

if high_risk_count > 0:
    print(f"\n⚠️  EVACUATION PRIORITY SHELTERS:")
    critical_shelters = gdf_shelters_risk[gdf_shelters_risk['high_risk_flag']]
    for idx, shelter in critical_shelters.iterrows():
        print(f"  • {shelter['name']} - {shelter['dynamic_risk']}")

# Save the map
output_file = 'output/typhoon_fungwong_risk_map.html'
m_lab2.save(output_file)

if os.path.exists(output_file):
    file_size = os.path.getsize(output_file)
    print(f"\n✓ Risk map saved to: {output_file}")
    print(f"✓ File size: {file_size/1024:.1f} KB")

print("\nDisplaying Typhoon Fung-wong risk map:")
display(m_lab2)

---

# 💭 My Reflection (fill in your answers)

### 1. How many lines of code did you change between LIVE and SIMULATION mode?

**Answer:** Only 1-2 lines of code needed to be changed between LIVE and SIMULATION mode. The key insight was using the same `parse_rainfall_json()` function for both data sources. The main difference was just the data source selection:

```python
# LIVE mode:
raw_data = fetch_cwa_api(api_key)

# SIMULATION mode:
with open('data/scenarios/fungwong_202511.json', 'r') as f:
    raw_data = json.load(f)
```

Both paths then use: `gdf_rainfall = parse_rainfall_json(raw_data)`

---

### 2. What happens if you forget to convert CRS before `sjoin`?

**Answer:** If you forget to convert CRS before `sjoin`, the spatial join will either fail completely or produce incorrect results. Different coordinate reference systems use different units and projections, so:

- **WGS84 (EPSG:4326)** uses degrees for coordinates
- **TWD97 (EPSG:3826)** uses meters for coordinates

When creating a 5km buffer, you need meter-based coordinates. Without CRS conversion, the buffer would be 5 degrees (≈555km) instead of 5km, making the spatial analysis meaningless. The `sjoin` operation would either throw an error or return incorrect matches.

---

### 3. Why does CWA use -998 instead of NaN or null?

**Answer:** CWA uses -998 as a NoData sentinel value for several practical reasons:

1. **Legacy Systems**: Many meteorological systems were designed when NaN/null support was limited
2. **Data Quality Control**: -998 is easily identifiable in quality control pipelines
3. **Compatibility**: Works across different programming languages and database systems
4. **Explicit Missing Data**: Makes it clear that data was intentionally marked as missing vs. system error
5. **Processing Efficiency**: Numeric operations can handle -998 without special null handling

This is common in environmental monitoring where data continuity is important.

---

### 4. During Typhoon Fung-wong, which shelter would you evacuate first and why?

**Answer:** I would evacuate **秀林鄉太魯閣避難所** first for these reasons:

1. **CRITICAL Risk Level**: 95.8 mm/hr rainfall (highest in the area)
2. **High Terrain Risk**: Located in mountainous terrain with 35.7° max slope
3. **High Elevation**: 285.7m elevation increases landslide risk
4. **Geographic Location**: Mountainous area prone to flash floods and landslides
5. **Limited Access**: Mountain roads may become impassable during heavy rain

The combination of extreme rainfall (>80 mm/hr) and HIGH terrain risk creates the most dangerous situation requiring immediate evacuation.

---

### 5. What challenges did you face in this lab? How did you solve them?

**Answer:** 

**Challenges faced:**

1. **JSON Format Variations**: CWA API, converted CSV, and XML downloads had different structures
   - **Solution**: Created `normalize_cwa_json()` function to detect and handle different formats

2. **Coordinate System Confusion**: Multiple coordinate systems (TWD67, WGS84) in same dataset
   - **Solution**: Added logic to specifically select WGS84 coordinates when available

3. **CRS Conversion Issues**: Initially used wrong geopandas function name
   - **Solution**: Used `gpd.points_from_xy()` instead of deprecated `points_from_coords()`

4. **Data Type Handling**: Precipitation values were sometimes strings, sometimes numbers
   - **Solution**: Added type checking and conversion with `isinstance()` checks

5. **Spatial Analysis Accuracy**: Needed meter-based coordinates for buffer operations
   - **Solution**: Ensured proper CRS conversion to EPSG:3826 before spatial operations

6. **Map Visualization**: Balancing visual clarity with data density
   - **Solution**: Implemented conditional styling, layer controls, and appropriate sizing

**Key Learning**: The importance of understanding coordinate systems and data cleaning in geospatial analysis. The mode-switcher architecture was particularly elegant for handling both real-time and simulation scenarios.